# How to train your AI: fra statistiske modeller til nevrale nettverk

Tech Day, Norges Bank, 6. november 2026. Simen Eide.

Vi skal prognosere **KPI-JAE neste måned**. Først med modeller fra SMART (AR, VAR, BVAR, faktormodell), bygget i PyTorch. Så gjør vi dem om til nevrale nettverk.

**Slik bruker du notebooken:** trykk ▶ på cellene ovenfra og ned. Du trenger ikke skrive kode. I hver oppgave endrer du tallene i skjemaet til høyre og kjører cellen på nytt. Vil du kode, er alt synlig.

Alle modeller følger de fire stegene: **data → modell → trening → beslutning**.

*Data: K2 - INTERNT. Ikke del datasettet utenfor Norges Bank.*

In [ ]:
# @title Maskinrom: kjør denne først { display-mode: "form" }
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

DATA = Path("data/forTecDay.xlsx")
if not DATA.exists():  # Colab: last opp fila
    from google.colab import files

    DATA = Path(next(iter(files.upload())))

raw = pd.read_excel(DATA).set_index("Time")
raw.index = pd.PeriodIndex([t.replace("M", "-") for t in raw.index], freq="M")
raw = raw.loc["2001-01":]
RATES = ["NAV.HL.TOT.PST.S"]  # ledighetsraten er allerede i prosent: bruk differansen
mm = 100 * np.log(raw.drop(columns=RATES)).diff()
mm[RATES] = raw[RATES].diff()

TARGET = "MSA_PCPIJAE"
DRIVERS = {
    "kronekurs": "I44",
    "oljepris": "DS.OILBRSV",
    "importpriser": "MSA_IPK",
    "innsatsvarer": "MSA_IPI",
    "ledighet": "NAV.HL.TOT.PST.S",
}
GROUPS = [c for c in raw if c.startswith(("MSA_PCPIJAE_25", "KPI.KPI"))]
y_all = mm[TARGET].shift(-1).rename("y")  # neste måneds endring
SPLIT = {
    "trening": ("2003-01", "2016-12"),
    "validering": ("2017-01", "2020-12"),
    "test": ("2021-01", "2100-01"),
}
LEADERBOARD: list[dict] = []


def lags(cols: list[str], p: int) -> pd.DataFrame:
    return pd.concat(
        {f"{c} (t-{k})": mm[c].shift(k) for c in cols for k in range(p)}, axis=1
    )


def make_x(
    p_target: int = 12,
    drivers: list[str] = (),
    p_drivers: int = 3,
    groups: bool = False,
    p_groups: int = 2,
) -> pd.DataFrame:
    parts = [lags([TARGET], p_target)]
    if drivers:
        parts.append(lags([DRIVERS[d] for d in drivers], p_drivers))
    if groups:
        parts.append(lags(GROUPS, p_groups))
    return pd.concat(parts, axis=1)


def tensors(x: pd.DataFrame):
    """Standardiser med treningsperioden, del i trening/validering/test, og ta med siste måned for prognosen."""
    tr = x.loc[slice(*SPLIT["trening"])]
    mu, sd = tr.mean(), tr.std().replace(0, 1)
    z = ((x - mu) / sd).fillna(0.0)
    out = {}
    for name, (a, b) in SPLIT.items():
        idx = y_all.loc[a:b].dropna().index
        out[name] = (
            torch.tensor(z.loc[idx].values, dtype=torch.float32),
            torch.tensor(y_all.loc[idx].values, dtype=torch.float32),
            idx,
        )
    out["siste"] = torch.tensor(z.iloc[[-1]].values, dtype=torch.float32)
    return out


def train(
    model: nn.Module,
    x: pd.DataFrame,
    epochs: int = 2000,
    lr: float = 0.01,
    weight_decay: float = 0.0,
    seed: int = 0,
):
    """Treningsløkka. Vi tar vare på vektene som traff best på valideringsperioden (early stopping)."""
    torch.manual_seed(seed)
    for m in model.modules():
        if hasattr(m, "reset_parameters"):
            m.reset_parameters()
    data = tensors(x)
    (xtr, ytr, _), (xva, yva, _) = data["trening"], data["validering"]
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist, best, state = {"trening": [], "validering": []}, float("inf"), None
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = ((model(xtr).squeeze(-1) - ytr) ** 2).mean()
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            val = ((model(xva).squeeze(-1) - yva) ** 2).mean().item()
        hist["trening"].append(loss.item() ** 0.5)
        hist["validering"].append(val**0.5)
        if val < best:
            best, state = val, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state)
    model.eval()
    return model, data, hist


def evaluate(name: str, model: nn.Module, data: dict, hist: dict | None = None) -> None:
    """RMSE i hver periode, en graf av prognosene i testperioden, og en rad på ledertavla."""
    res = {"modell": name, "parametere": sum(p.numel() for p in model.parameters())}
    with torch.no_grad():
        for part in ["trening", "validering", "test"]:
            x, y, idx = data[part]
            p = model(x).squeeze(-1)
            res[part] = ((p - y) ** 2).mean().item() ** 0.5
        nxt = model(data["siste"]).item()
    res["prognose neste mnd"] = nxt
    LEADERBOARD.append(res)
    x, y, idx = data["test"]
    with torch.no_grad():
        p = model(x).squeeze(-1).numpy()
    _, axes = plt.subplots(1, 2 if hist else 1, figsize=(14 if hist else 9, 3.6))
    ax = axes[0] if hist else axes
    t = idx.to_timestamp()
    ax.plot(t, y.numpy(), color="black", label="Faktisk")
    ax.plot(t, p, color="tab:red", label=name)
    ax.set_title(f"Testperioden: RMSE {res['test']:.3f}")
    ax.set_ylabel("Endring i KPI-JAE, %")
    ax.legend()
    if hist:
        axes[1].plot(hist["trening"], label="Trening")
        axes[1].plot(hist["validering"], label="Validering")
        axes[1].set_title("Treningsløkka: feil per runde")
        axes[1].set_xlabel("Runde")
        axes[1].set_ylabel("RMSE")
        axes[1].set_ylim(0, max(0.6, min(hist["validering"]) * 2))
        axes[1].legend()
    plt.tight_layout()
    plt.show()
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in res.items()})


def leaderboard() -> pd.DataFrame:
    return (
        pd.DataFrame(LEADERBOARD)
        .drop_duplicates("modell", keep="last")
        .sort_values("validering")
        .round(3)
        .reset_index(drop=True)
    )


def rmse(p, part: str = "test") -> float:
    a, b = SPLIT[part]
    y = y_all.loc[a:b].dropna()
    return float(np.sqrt(np.mean((np.asarray(p.loc[y.index]) - y.values) ** 2)))


# Enkle referanser
for name, p in {
    "Naiv: samme som forrige måned": mm[TARGET],
    "Snitt siste 12 mnd": mm[TARGET].rolling(12).mean(),
}.items():
    LEADERBOARD.append(
        {
            "modell": name,
            "parametere": 0,
            **{k: rmse(p, k) for k in SPLIT},
            "prognose neste mnd": float(p.iloc[-1]),
        }
    )
print(
    f"Data: {raw.index[0]} til {raw.index[-1]}, {raw.shape[1]} serier. Siste kjente måned: {raw.index[-1]}."
)
leaderboard()

## Data

KPI-JAE, sesongjustert. Vi prognoserer **endringen fra én måned til neste**, i prosent. Ett datapunkt er én måned.

Vi deler historien i tre: **trening** (2003-2016) der modellen lærer, **validering** (2017-2020) der vi velger modell, og **test** (2021-) der vi ser hvor godt det gikk. Testperioden har inflasjonssjokket.

In [ ]:
_, ax = plt.subplots(figsize=(14, 3.6))
s = mm[TARGET].loc["2003":]
for (name, (a, b)), c in zip(SPLIT.items(), ["tab:blue", "tab:orange", "tab:red"]):
    ax.plot(s.loc[a:b].index.to_timestamp(), s.loc[a:b], color=c, label=name)
ax.set_ylabel("Endring i KPI-JAE, %")
ax.legend()
plt.show()

## Oppgave 1: AR-modellen er et nevralt nett med ett lag

SMART har mange AR-modeller: neste måned er en vektet sum av de `P` siste månedene. I PyTorch er det `nn.Linear(P, 1)`: `P` noder inn, én ut, og vektene er strekene mellom dem.

**Prøv:** endre `P` (antall måneder modellen ser tilbake). Hva skjer med testfeilen? Hvilke måneder får mest vekt?

In [ ]:
P = 12  # @param {type:"slider", min:1, max:24, step:1}
EPOCHS = 2000  # @param {type:"integer"}
LR = 0.01  # @param {type:"number"}

x_ar = make_x(p_target=P)
ar, data, hist = train(nn.Linear(P, 1), x_ar, epochs=EPOCHS, lr=LR)
evaluate(f"AR({P})", ar, data, hist)

# Samme svar som minste kvadraters metode (OLS)?
xtr, ytr, _ = data["trening"]
ols = torch.linalg.lstsq(
    torch.cat([xtr, torch.ones(len(xtr), 1)], 1), ytr.unsqueeze(1)
).solution.squeeze()
plt.figure(figsize=(9, 3))
plt.bar(
    np.arange(P) - 0.2, ar.weight.detach().squeeze().numpy(), width=0.4, label="PyTorch"
)
plt.bar(np.arange(1, P + 1) + 0.2, ols[:-1].numpy(), width=0.4, label="OLS")
plt.xlabel("Måneder tilbake")
plt.ylabel("Vekt")
plt.title("Vektene i AR-modellen")
plt.legend()
plt.show()

## Oppgave 2: VAR og en prior

En VAR-likning tar med andre serier: kronekurs, olje, importpriser, ledighet. Med `MED_GRUPPER` tar vi også med 53 delindekser av KPI. Da får modellen flere parametere enn vi har måneder å trene på.

**Prøv:**
1. Skru på `MED_GRUPPER` med `WEIGHT_DECAY = 0`. Se på grafen til høyre: treningsfeilen går ned, valideringsfeilen går opp. Det er **overtilpasning**.
2. Øk `WEIGHT_DECAY`. Det er det samme som en Bayesiansk prior (som i BVAR): vektene trekkes mot null. Når blir testfeilen best?

In [ ]:
kronekurs = True  # @param {type:"boolean"}
oljepris = True  # @param {type:"boolean"}
importpriser = True  # @param {type:"boolean"}
innsatsvarer = False  # @param {type:"boolean"}
ledighet = False  # @param {type:"boolean"}
MED_GRUPPER = False  # @param {type:"boolean"}
WEIGHT_DECAY = 0.0  # @param [0.0, 0.01, 0.1, 0.3, 1.0, 3.0] {type:"raw"}

chosen = [
    k
    for k, v in {
        "kronekurs": kronekurs,
        "oljepris": oljepris,
        "importpriser": importpriser,
        "innsatsvarer": innsatsvarer,
        "ledighet": ledighet,
    }.items()
    if v
]
x_var = make_x(p_target=12, drivers=chosen, groups=MED_GRUPPER)
print(
    f"{x_var.shape[1]} input-variabler, {len(y_all.loc[slice(*SPLIT['trening'])].dropna())} måneder å trene på"
)
var, data, hist = train(nn.Linear(x_var.shape[1], 1), x_var, weight_decay=WEIGHT_DECAY)
evaluate(
    f"VAR ({', '.join(chosen) or 'ingen drivere'}{', grupper' if MED_GRUPPER else ''}, wd={WEIGHT_DECAY})",
    var,
    data,
    hist,
)

## Oppgave 3: fra faktormodell til nevralt nett

En faktormodell komprimerer mange serier til noen få **faktorer**, og prognoserer fra dem. I PyTorch er det to lag: `Linear(n, R)` og så `Linear(R, 1)`. Det midterste laget er et **skjult lag**, en flaskehals.

Legg til en ikke-lineær funksjon (ReLU) i det skjulte laget, og vi har et ekte nevralt nettverk.

**Prøv:** endre antall faktorer `R`, bytt `AKTIVERING` til ReLU, og legg til `DROPOUT` (skru av tilfeldige noder under trening, en annen type regularisering).

In [ ]:
R = 3  # @param {type:"slider", min:1, max:64, step:1}
AKTIVERING = "ingen (faktormodell)"  # @param ["ingen (faktormodell)", "ReLU", "Tanh"]
DROPOUT = 0.0  # @param {type:"slider", min:0, max:0.8, step:0.1}
WEIGHT_DECAY_3 = 0.1  # @param [0.0, 0.01, 0.1, 0.3, 1.0, 3.0] {type:"raw"}

x_big = make_x(p_target=12, drivers=list(DRIVERS), groups=True)
act = {"ingen (faktormodell)": nn.Identity(), "ReLU": nn.ReLU(), "Tanh": nn.Tanh()}[
    AKTIVERING
]
net = nn.Sequential(
    nn.Linear(x_big.shape[1], R), act, nn.Dropout(DROPOUT), nn.Linear(R, 1)
)
print(net)
net, data, hist = train(net, x_big, lr=0.003, weight_decay=WEIGHT_DECAY_3)
evaluate(
    f"{AKTIVERING}, R={R}, dropout={DROPOUT}, wd={WEIGHT_DECAY_3}", net, data, hist
)

## Oppgave 4: slå AR

Ingen av modellene over slår AR-modellen lett. Kan du? Bygg din egen modell under. Alt er lov, men velg modell på **valideringsfeilen**, ikke testfeilen. Ellers jukser du med framtiden.

Idéer: færre eller flere lag, bare noen av gruppene, kortere historikk (`p_target`), flere skjulte lag, kombiner prognosene fra flere modeller (som SMART gjør).

In [ ]:
NAVN = "Min modell"  # @param {type:"string"}

x_mine = make_x(p_target=6, drivers=["kronekurs", "importpriser"], groups=False)
mine = nn.Sequential(
    nn.Linear(x_mine.shape[1], 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
mine, data, hist = train(mine, x_mine, lr=0.003, weight_decay=0.3)
evaluate(NAVN, mine, data, hist)

## Ledertavla

Sortert etter valideringsfeil. Den som er best på validering, er den vi ville valgt. Stemmer det med test?

In [ ]:
leaderboard()

## Beslutning: hva blir KPI-JAE i oktober?

Modellen som var best på validering, gir en prognose for oktober. SSB publiserer tallet 10. november.

In [ ]:
lb = leaderboard()
best = lb.iloc[0]
level = raw[TARGET]
oct_level = level.iloc[-1] * np.exp(best["prognose neste mnd"] / 100)
yoy = 100 * (oct_level / level.loc[str(level.index[-1] - 11)] - 1)
print(f"Beste modell på validering: {best['modell']}")
print(
    f"Prognose for {level.index[-1] + 1}: {best['prognose neste mnd']:+.2f} % fra måneden før, {yoy:.1f} % fra samme måned i fjor (sesongjustert)"
)